# 05 – Bestimmung der Themenanzahl

Hier wird die Zahl der Themen für LDA und NMF systematisch über den Coherence Score bestimmt. Für einen Bereich von k werden beide Verfahren trainiert, bewertet und verglichen.

**Eingabe:**
`data/tokens_final.pkl` (19.807 Beschwerden als Wortlisten, dieselbe Datenbasis wie in `03` und `04`)

**Ausgabe:**
Coherence Scores je k, Diagramm und die festgelegte Themenanzahl

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.corpora import Dictionary

In [ ]:
# Laden
path = Path("data/tokens_final.pkl")
if not path.exists():
    raise FileNotFoundError("data/tokens_final.pkl fehlt, bitte zuerst 02_vorverarbeitung ausführen.")

df = pd.read_pickle(path)
print(df.shape)

In [ ]:
# Bag-of-Words
cv = CountVectorizer(analyzer=lambda tokens: tokens)
X_bow = cv.fit_transform(df["tokens_filtered"])

# TF-IDF
tv = TfidfVectorizer(analyzer=lambda tokens: tokens)
X_tfidf = tv.fit_transform(df["tokens_filtered"])

# gensim: Wörterbuch und Corpus
dictionary = Dictionary(df["tokens_filtered"])
corpus = [dictionary.doc2bow(toks) for toks in df["tokens_filtered"]]

In [ ]:
# Kontrolle
summe_gensim = sum(c for doc in corpus for _, c in doc)

print("BoW:", X_bow.shape, "| Einträge ungleich 0:", X_bow.nnz, "| Summe:", X_bow.sum())
print("TF-IDF:", X_tfidf.shape, "| Einträge ungleich 0:", X_tfidf.nnz)
print("gensim: Wörterbuch:", len(dictionary), "| Summe:", summe_gensim)

assert X_bow.shape[0] == len(df)
assert X_bow.shape == X_tfidf.shape
assert X_bow.nnz == X_tfidf.nnz
assert len(dictionary) == X_bow.shape[1]
assert summe_gensim == X_bow.sum() == df["n_tokens"].sum()